---
title: "Collaboration Network"
date: today
date-format: "DD MMMM YYYY"
language:
  title-block-published: "Updated"
jupyter: "card-lab"
execute: true
format: html
---

In [1]:
#| echo: false
#| eval: true
#| output: false

"""
This notebook builds a collaboration network from CARD Lab Zotero records in [card-lab.github.io/files/zotero-items.json](card-lab.github.io/files/zotero-items.json), following the same core methodology as the Dimensions organizations collaboration network cookbook:

- define entities (authors)
- connect entities by co-occurrence on publications
- weight edges by collaboration frequency
- visualize the resulting weighted graph

The notebook emphasizes CARD core members and their direct collaborators.
"""

from __future__ import annotations

import json
import math
import re
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import networkx as nx
import pandas as pd
import plotly.graph_objects as go

# Resolve paths relative to the notebook's location.
# Try __file__ first (works with Quarto), then fall back to searching for card-lab.github.io in the directory tree.
def _resolve_notebook_dir() -> Path:
    try:
        return Path(__file__).parent.resolve()
    except NameError:
        # Fallback: search parent directories for card-lab.github.io/research
        cwd = Path.cwd().resolve()
        for parent in [cwd] + list(cwd.parents):
            candidate = parent / "card-lab.github.io" / "research"
            if candidate.exists():
                return candidate
            if parent.name == "research" and (parent.parent / "files").exists():
                return parent
        return cwd

NOTEBOOK_DIR = _resolve_notebook_dir()
CARD_LAB_GITHUB_IO = NOTEBOOK_DIR.parent

# Path to Zotero export and related resources
ZOTERO_JSON_PATH = CARD_LAB_GITHUB_IO / "files" / "zotero-items.json"
TIMELINE_XLSX_PATH = CARD_LAB_GITHUB_IO / "private" / "CARD Group Timeline.xlsx"
PEOPLE_CURRENT_DIR = CARD_LAB_GITHUB_IO / "people" / "current"
PEOPLE_ALUMNI_DIR = CARD_LAB_GITHUB_IO / "people" / "alumni"
PEOPLE_PI_FILE = CARD_LAB_GITHUB_IO / "people" / "principal-investigator.qmd"
PI_DISPLAY_NAME = "E. J. Payton"


def _extract_title_from_qmd(path: Path) -> str | None:
    text = path.read_text(encoding="utf-8", errors="ignore")
    if not text.startswith("---"):
        return None
    parts = text.split("---", 2)
    if len(parts) < 3:
        return None
    frontmatter = parts[1]
    m = re.search(r"^\s*title\s*:\s*['\"]?(.*?)['\"]?\s*$", frontmatter, flags=re.MULTILINE)
    if not m:
        return None
    title = " ".join(m.group(1).strip().split())
    return title if title else None


def _load_people_page_names() -> set[str]:
    names: set[str] = set()
    for d in [PEOPLE_CURRENT_DIR, PEOPLE_ALUMNI_DIR]:
        if not d.exists():
            continue
        for p in sorted(d.glob("*.qmd")):
            name = _extract_title_from_qmd(p)
            if name:
                names.add(name)

    if PEOPLE_PI_FILE.exists():
        pi_name = _extract_title_from_qmd(PEOPLE_PI_FILE)
        if pi_name:
            # PI page title is not a personal name, so map to canonical display name.
            if pi_name.strip().lower() == "about the pi":
                names.add(PI_DISPLAY_NAME)
            else:
                names.add(pi_name)

    return names


def _load_timeline_names() -> set[str]:
    if not TIMELINE_XLSX_PATH.exists():
        return set()
    try:
        df_people = pd.read_excel(TIMELINE_XLSX_PATH, sheet_name="People")
        if "Display Name" not in df_people.columns:
            return set()
        names = {
            " ".join(str(x).strip().split())
            for x in df_people["Display Name"].dropna().tolist()
            if str(x).strip()
        }
        return names
    except Exception:
        return set()


people_page_names = _load_people_page_names()
timeline_names = _load_timeline_names()

# Core list requested: timeline roster equivalent to people pages.
# We use people pages as the authoritative set of members with published profile pages.
CARD_CORE_AUTHORS = set(people_page_names) if people_page_names else set(timeline_names)

if not ZOTERO_JSON_PATH.exists():
    raise FileNotFoundError(f"Could not find {ZOTERO_JSON_PATH.resolve()}")

print(f"Using Zotero file: {ZOTERO_JSON_PATH}")
print(f"People page names found: {len(people_page_names)}")
print(f"Timeline names found: {len(timeline_names)}")
print(f"CARD core display-name list size: {len(CARD_CORE_AUTHORS)}")
if people_page_names and timeline_names:
    only_people = sorted(people_page_names - timeline_names)
    only_timeline = sorted(timeline_names - people_page_names)
    print(f"People-only names not in timeline: {only_people[:3]}{' ...' if len(only_people) > 3 else ''}")
    print(f"Timeline-only names not in people pages: {only_timeline[:3]}{' ...' if len(only_timeline) > 3 else ''}")

Using Zotero file: /Users/paytone/Library/CloudStorage/OneDrive-UniversityofCincinnati/Code/card-lab/card-lab.github.io/files/zotero-items.json
People page names found: 41
Timeline names found: 41
CARD core display-name list size: 41
People-only names not in timeline: ['E. J. Payton']
Timeline-only names not in people pages: ['Eric Payton']


/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



In [2]:
#| echo: false
#| eval: true
#| output: false

KNOWN_ALIAS_TO_CANONICAL = {
    "gunther eggeler": "G. Eggeler",
    "g. eggeler": "G. Eggeler",
    "yuval noiman": "Y. Noiman",
    "y. noiman": "Y. Noiman",
    "sravya josyula": "S. Josyula",
    "s. josyula": "S. Josyula",
    "eric john payton": "E. Payton",
    "eric j. payton": "E. Payton",
    "eric payton": "E. Payton",
    "e. j. payton": "E. Payton",
    "e.j. payton": "E. Payton",
    "e. payton": "E. Payton",
}


def _clean_space(text: str) -> str:
    return " ".join(str(text or "").strip().split())


def _normalize_text_key(text: str) -> str:
    t = _clean_space(text).lower()
    t = re.sub(r"\s+", " ", t)
    return t


def _tokenize_first_names(first_name: str) -> list[str]:
    first_name = _clean_space(first_name.replace("-", " "))
    return [t for t in first_name.split() if t]


def _initials_from_tokens(tokens: list[str]) -> list[str]:
    initials = []
    for tok in tokens:
        t = tok.strip()
        if not t:
            continue
        # Keep compact initial-like tokens, else convert to first-letter initial.
        if t.endswith(".") and len(t) <= 3:
            initials.append(t[0].upper() + ".")
        else:
            initials.append(t[0].upper() + ".")
    return initials


def _parse_creator_name(creator: dict) -> tuple[str, str, str]:
    """Return (first_name, last_name, raw_full_name)."""
    first = _clean_space(creator.get("firstName", ""))
    last = _clean_space(creator.get("lastName", ""))

    if first or last:
        raw = _clean_space(f"{first} {last}")
        return first, last, raw

    raw = _clean_space(creator.get("name", ""))
    if not raw:
        return "", "", ""

    parts = raw.split()
    if len(parts) == 1:
        return "", parts[0], raw
    return " ".join(parts[:-1]), parts[-1], raw


def _build_ambiguity_index(items: list[dict]) -> set[tuple[str, str]]:
    """Find (last, first-initial) pairs mapping to multiple full first names."""
    observed: dict[tuple[str, str], set[str]] = defaultdict(set)

    for item in items:
        for creator in item.get("data", {}).get("creators", []):
            if str(creator.get("creatorType", "")).lower() not in ("", "author"):
                continue
            first, last, _ = _parse_creator_name(creator)
            last_key = _normalize_text_key(last)
            if not last_key:
                continue

            tokens = _tokenize_first_names(first)
            if not tokens:
                continue
            first_initial = tokens[0][0].lower()

            # Use first token only for ambiguity test (Eric vs Ethan etc.).
            first_token_key = _normalize_text_key(tokens[0].strip("."))
            if len(first_token_key) > 1:
                observed[(last_key, first_initial)].add(first_token_key)

    return {k for k, vals in observed.items() if len(vals) > 1}


def normalize_author_name(creator: dict, ambiguous_initial_last: set[tuple[str, str]]) -> str | None:
    """Return robust canonical author name with abbreviation-aware alias handling."""
    creator_type = str(creator.get("creatorType", "")).lower()
    if creator_type and creator_type != "author":
        return None

    first, last, raw_full = _parse_creator_name(creator)
    if not last:
        return None

    raw_key = _normalize_text_key(raw_full)
    if raw_key in KNOWN_ALIAS_TO_CANONICAL:
        return KNOWN_ALIAS_TO_CANONICAL[raw_key]

    tokens = _tokenize_first_names(first)
    initials = _initials_from_tokens(tokens)

    # If we don't have a first token, keep last-name-only fallback.
    if not initials:
        return last

    first_initial = initials[0].replace(".", "").lower()
    key = (_normalize_text_key(last), first_initial)

    # Prefer compact canonical form "E. Last" unless ambiguous in corpus.
    if key not in ambiguous_initial_last:
        canonical = f"{initials[0]} {last}"
    else:
        # In ambiguous cases, preserve extra initials if present.
        canonical = f"{' '.join(initials)} {last}"

    canonical = _clean_space(canonical)
    return KNOWN_ALIAS_TO_CANONICAL.get(_normalize_text_key(canonical), canonical)


with ZOTERO_JSON_PATH.open("r", encoding="utf-8") as f:
    zotero_items = json.load(f)

AMBIGUOUS_INITIAL_LAST = _build_ambiguity_index(zotero_items)

records: list[dict] = []
for item in zotero_items:
    data = item.get("data", {})
    creators = data.get("creators", [])
    authors = [normalize_author_name(c, AMBIGUOUS_INITIAL_LAST) for c in creators]
    authors = sorted({a for a in authors if a})

    if len(authors) < 2:
        continue

    records.append(
        {
            "item_key": data.get("key"),
            "title": data.get("title", ""),
            "date": data.get("date", ""),
            "publication": data.get("publicationTitle", ""),
            "doi": str(data.get("DOI", "") or "").strip(),
            "authors": authors,
            "raw_data": data,
        }
    )

papers_df = pd.DataFrame(records)
print(f"Multi-author items included: {len(papers_df)}")
print(f"Ambiguous initial+last pairs requiring extra initials: {len(AMBIGUOUS_INITIAL_LAST)}")

papers_df[["item_key", "title", "authors"]].head(3)

Multi-author items included: 23
Ambiguous initial+last pairs requiring extra initials: 0


,item_key,title,authors
0,5ZYQZ36M,Computational discovery of medium-entropy SMAs...,"[A. Oladipo, E. Payton, J. Maile, N. Simpson, ..."
1,6YIC2KG7,Improved Fe-Ni-P thermodynamic models with rev...,"[E. Payton, M. Steiner, U. Ochieze, Z. Miller]"
2,X5EIIH4M,Physics-augmented machine learning for predict...,"[E. Payton, M. Steiner, S. Josyula, Y. Noiman]"


In [3]:
#| echo: false
#| eval: true
#| output: false

# Build weighted co-authorship graph.

G = nx.Graph()

for _, row in papers_df.iterrows():
    authors = row["authors"]
    title = row["title"]
    item_key = row["item_key"]

    # Add node paper counts.
    for author in authors:
        if author not in G:
            G.add_node(author, papers=0)
        G.nodes[author]["papers"] += 1

    # Add weighted edges per co-author pair.
    for a, b in combinations(authors, 2):
        if G.has_edge(a, b):
            G[a][b]["weight"] += 1
            G[a][b]["papers"].append(item_key)
        else:
            G.add_edge(a, b, weight=1, papers=[item_key], sample_title=title)


def _canonical_from_display_name(name: str) -> str | None:
    parts = str(name).strip().split()
    if not parts:
        return None
    if len(parts) == 1:
        creator = {"creatorType": "author", "lastName": parts[0]}
    else:
        creator = {
            "creatorType": "author",
            "firstName": " ".join(parts[:-1]),
            "lastName": parts[-1],
        }
    return normalize_author_name(creator, AMBIGUOUS_INITIAL_LAST)


def _initial_last_from_display_name(name: str) -> str | None:
    parts = str(name).strip().split()
    if len(parts) < 2:
        return None
    return f"{parts[0][0].upper()}. {parts[-1]}"


def _display_name_tokens(name: str) -> tuple[str, str]:
    parts = [p for p in str(name).strip().split() if p]
    if not parts:
        return "", ""
    first = parts[0]
    last = parts[-1]
    return first, last


def _node_last_name(node_name: str) -> str:
    parts = [p for p in str(node_name).strip().split() if p]
    return parts[-1].lower() if parts else ""


# Build reverse index for graph-node last names, used by fallback matching.
last_name_to_nodes: dict[str, set[str]] = defaultdict(set)
for node in G.nodes:
    ln = _node_last_name(node)
    if ln:
        last_name_to_nodes[ln].add(node)


# Resolve each roster display name to one or more graph node names.
resolved_core_nodes: set[str] = set()
unresolved_display_names: list[str] = []

for display_name in sorted(CARD_CORE_AUTHORS):
    candidates: set[str] = set()

    # Candidate generation from existing canonicalization logic.
    for c in [
        _canonical_from_display_name(display_name),
        _initial_last_from_display_name(display_name),
    ]:
        if not c:
            continue
        candidates.add(c)
        candidates.add(KNOWN_ALIAS_TO_CANONICAL.get(_normalize_text_key(c), c))
        candidates.add(KNOWN_ALIAS_TO_CANONICAL.get(c, c))
        candidates.add(KNOWN_ALIAS_TO_CANONICAL.get(c.replace(" ", ""), c))

    # Keep candidates that actually exist as graph nodes.
    matched = {c for c in candidates if c in G.nodes}

    # Fallback 1: unique-last-name matching catches cases like
    # Tobi Oriola (people page) vs A. Oriola (publication metadata).
    _, last = _display_name_tokens(display_name)
    if not matched and last:
        same_last = last_name_to_nodes.get(last.lower(), set())
        if len(same_last) == 1:
            matched.update(same_last)

    # Fallback 2: if there are multiple same-last-name nodes, keep those
    # sharing first initial when possible.
    if not matched and last:
        first, _ = _display_name_tokens(display_name)
        same_last = last_name_to_nodes.get(last.lower(), set())
        if first and same_last:
            fi = first[0].upper()
            matched.update({n for n in same_last if n.strip().upper().startswith(fi + ".")})

    if matched:
        resolved_core_nodes.update(matched)
    else:
        unresolved_display_names.append(display_name)

CARD_CORE_AUTHORS_CANONICAL = resolved_core_nodes
if not CARD_CORE_AUTHORS_CANONICAL:
    CARD_CORE_AUTHORS_CANONICAL = {"E. Payton"}

# Flag CARD core members and derive a CARD-focused subgraph.
for author in G.nodes:
    G.nodes[author]["is_card_core"] = author in CARD_CORE_AUTHORS_CANONICAL

card_nodes = {n for n, d in G.nodes(data=True) if d.get("is_card_core", False)}
if not card_nodes:
    raise ValueError(
        "No CARD core authors found in graph. Verify timeline/people parsing and aliases."
    )

neighbors = set(card_nodes)
for n in card_nodes:
    neighbors.update(G.neighbors(n))

H = G.subgraph(neighbors).copy()
print(f"Canonical CARD core author names used: {len(CARD_CORE_AUTHORS_CANONICAL)}")
print(f"Unresolved CARD display names (no publication-node match): {len(unresolved_display_names)}")
if unresolved_display_names:
    print(f"Sample unresolved: {unresolved_display_names[:8]}")
print(f"Full graph: {G.number_of_nodes()} authors, {G.number_of_edges()} collaborations")
print(f"CARD-focused graph: {H.number_of_nodes()} authors, {H.number_of_edges()} collaborations")

# Targeted status check for commonly referenced members.
for ln in ["jestude", "dursun", "sun", "oriola"]:
    nodes = sorted(last_name_to_nodes.get(ln, set()))
    core_nodes = [n for n in nodes if G.nodes[n].get("is_card_core", False)]
    print(f"{ln}: nodes={nodes}, core={core_nodes}")

Canonical CARD core author names used: 20
Unresolved CARD display names (no publication-node match): 21
Sample unresolved: ['Aaron McMillen', 'Ben Honeck', 'Cameron Gulley', 'Chris Le', 'Darrian Mathis', 'Emilie Maddox', 'Emily Crawley', 'Gwynn Pitz']
Full graph: 68 authors, 252 collaborations
CARD-focused graph: 68 authors, 252 collaborations
jestude: nodes=['Z. Jestude'], core=['Z. Jestude']
dursun: nodes=['E. Dursun'], core=['E. Dursun']
sun: nodes=['L. Sun'], core=['L. Sun']
oriola: nodes=['A. Oriola'], core=['A. Oriola']


In [4]:
#| echo: false
#| eval: true
#| output: false

# Summary table of top collaborators (by weighted degree) in CARD-focused network.
weighted_degree = dict(H.degree(weight="weight"))
summary = (
    pd.DataFrame(
        {
            "author": list(weighted_degree.keys()),
            "weighted_collaboration_degree": list(weighted_degree.values()),
            "paper_count": [H.nodes[a].get("papers", 0) for a in weighted_degree.keys()],
            "is_card_core": [H.nodes[a].get("is_card_core", False) for a in weighted_degree.keys()],
        }
    )
    .sort_values(["is_card_core", "weighted_collaboration_degree", "paper_count"], ascending=[False, False, False])
    .reset_index(drop=True)
)

summary.head(20)

,author,weighted_collaboration_degree,paper_count,is_card_core
0,E. Payton,95,23,True
1,U. Ochieze,20,4,True
2,S. Josyula,19,4,True
3,J. Maile,18,4,True
4,Y. Noiman,12,3,True
5,A. Nguyen,12,3,True
6,A. Oriola,12,3,True
7,A. Oladipo,8,2,True
8,L. Sun,7,1,True
9,D. Timberlake,5,2,True


In [5]:
#| echo: false
#| eval: true
#| output: true

# Interactive 3D collaboration network visualization (drag to rotate).
# Nodes: authors
# Edges: co-authorship, weighted by number of shared publications


pos3d = nx.spring_layout(H, dim=3, k=1.1 / math.sqrt(max(1, H.number_of_nodes())), seed=42)

edge_x, edge_y, edge_z = [], [], []
for u, v, _ in H.edges(data=True):
    x0, y0, z0 = pos3d[u]
    x1, y1, z1 = pos3d[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    edge_z += [z0, z1, None]

edge_trace = go.Scatter3d(
    x=edge_x,
    y=edge_y,
    z=edge_z,
    mode="lines",
    line=dict(width=4, color="rgba(120,120,120,0.6)"),
    hoverinfo="none",
)

node_x, node_y, node_z = [], [], []
node_text, node_size, node_color = [], [], []

for n, attrs in H.nodes(data=True):
    x, y, z = pos3d[n]
    papers = attrs.get("papers", 0)
    wdeg = weighted_degree.get(n, 0)
    is_core = attrs.get("is_card_core", False)

    node_x.append(x)
    node_y.append(y)
    node_z.append(z)
    node_size.append(6 + 3 * math.sqrt(max(1, wdeg)))
    node_color.append("#d62728" if is_core else "#1f77b4")
    node_text.append(
        f"<b>{n}</b><br>"
        f"CARD core: {'Yes' if is_core else 'No'}<br>"
        f"Papers: {papers}<br>"
        f"Weighted degree: {wdeg}"
    )

node_trace = go.Scatter3d(
    x=node_x,
    y=node_y,
    z=node_z,
    mode="markers+text",
    text=[n for n in H.nodes()],
    textposition="top center",
    hovertemplate="%{customdata}<extra></extra>",
    customdata=node_text,
    marker=dict(size=node_size, color=node_color, line=dict(width=0.8, color="white"), opacity=0.95),
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title="CARD Lab Collaboration Network",
    showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=0, r=0, t=50, b=0),
    scene=dict(
        bgcolor="white",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
        dragmode="orbit",
    ),
)
# Add very slow auto-rotation by animating the camera eye around the z-axis.
n_frames = 240
radius = 1.9
z_eye = 0.9
fig.frames = [
    go.Frame(layout=dict(scene_camera=dict(eye=dict(
        x=radius * math.cos(2 * math.pi * i / n_frames),
        y=radius * math.sin(2 * math.pi * i / n_frames),
        z=z_eye,
    ))))
    for i in range(n_frames)
]
fig.update_layout(scene_camera=dict(eye=dict(x=radius, y=0, z=z_eye)))
fig.show()

In [6]:
#| echo: false
#| eval: true
#| output: false

# Affiliation collaboration network (first hop from CARD affiliations).
# Strategy:
# 1) Pull author-level affiliations from DOI metadata (Crossref + OpenAlex + DataCite).
# 2) Pull work-level institutions from DOI metadata (fallback when author matching misses).
# 3) Pull creator-level affiliation fields if present in Zotero creators.
# 4) Parse potential affiliation lines from Zotero `extra` field.
# 5) Apply author->affiliation overrides for known CARD collaborators.
# 6) Canonicalize affiliation names with static + dynamic alias consolidation.
# 7) Build weighted affiliation co-occurrence graph and keep first-hop nodes.

import requests
import urllib.parse



CARD_SEED_AFFILIATIONS = {
    "University of Cincinnati",
}

# Manual overrides can be extended as needed to improve affiliation completeness.
AUTHOR_AFFILIATION_OVERRIDES = {
    "E. Payton": {"University of Cincinnati"},
}

TARGET_DIAGNOSTIC_DOI = "10.21949/rfzv-6285"

INSTITUTION_KEYWORDS = {
    "university",
    "institute",
    "institut",
    "college",
    "school",
    "laboratory",
    "lab",
    "center",
    "centre",
    "academy",
    "polytechnic",
    "hospital",
    "cnrs",
    "afrl",
    "company",
    "corporation",
    "corp",
    "inc",
    "llc",
    "ltd",
    "gmbh",
    "ag",
    "sa",
    "technologies",
    "technology",
    "materials",
    "systems",
    "resources",
    "engineering",
}

COUNTRY_OR_REGION_TOKENS = {
    "usa",
    "u.s.a",
    "united states",
    "uk",
    "u.k",
    "united kingdom",
    "germany",
    "france",
    "china",
    "india",
    "nigeria",
    "australia",
    "new zealand",
    "turkiye",
    "turkey",
}

# Base alias table for known recurring institution variants.
AFFILIATION_ALIAS_STATIC: dict[str, str] = {
    "university of cincinnati": "University of Cincinnati",
    "department of mechanical and materials engineering university of cincinnati": "University of Cincinnati",
    "centre national de la recherche scientifique": "CNRS",
    "cnrs": "CNRS",
    "universite paris sciences et lettres": "PSL University",
    "universite paris sciences et lettres psl university": "PSL University",
    "psl university": "PSL University",
    "ruhr university bochum": "Ruhr University Bochum",
    "institute for materials ruhr university bochum": "Ruhr University Bochum",
    "wright patterson air force base": "Wright-Patterson Air Force Base",
    "united states air force research laboratory": "Air Force Research Laboratory",
    "air force research laboratory": "Air Force Research Laboratory",
}

# Runtime-learned aliases. This updates itself as new variants are encountered.
AFFILIATION_ALIAS_DYNAMIC: dict[str, str] = {}

# Track the most descriptive raw affiliation string encountered per canonical name.
AFFILIATION_FULL_ADDRESS: dict[str, str] = {}


def _normalize_doi(doi: str) -> str:
    doi = (doi or "").strip().lower()
    if doi.startswith("https://doi.org/"):
        doi = doi.replace("https://doi.org/", "", 1)
    if doi.startswith("http://doi.org/"):
        doi = doi.replace("http://doi.org/", "", 1)
    return doi


def _affiliation_key(text: str) -> str:
    t = str(text or "").lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def _register_alias(raw: str, canonical: str) -> None:
    key = _affiliation_key(raw)
    if key and canonical:
        AFFILIATION_ALIAS_DYNAMIC[key] = canonical


def _full_address_score(text: str) -> tuple[int, int, int, int]:
    s = str(text or "")
    low = s.lower()
    has_department = int(any(k in low for k in ["department", "dept", "school of", "faculty of", "division of", "laboratory", "lab"]))
    has_address_detail = int(bool(re.search(r"\b\d{3,}\b", low) or any(k in low for k in ["street", "st.", "avenue", "ave", "road", "rd", "building", "campus"])))
    return (has_department, has_address_detail, s.count(","), len(s))


def _register_full_affiliation(raw: str, canonical: str) -> None:
    if not canonical:
        return
    clean_raw = " ".join(str(raw or "").replace(";", ",").split())
    if not clean_raw:
        return
    prev = AFFILIATION_FULL_ADDRESS.get(canonical, "")
    if _full_address_score(clean_raw) > _full_address_score(prev):
        AFFILIATION_FULL_ADDRESS[canonical] = clean_raw


def _canonical_from_given_family(given: str, family: str) -> str | None:
    creator = {"creatorType": "author", "firstName": given or "", "lastName": family or ""}
    return normalize_author_name(creator, AMBIGUOUS_INITIAL_LAST)


def _canonical_from_display(display_name: str) -> str | None:
    tokens = [t for t in str(display_name or "").strip().split() if t]
    if not tokens:
        return None
    if len(tokens) == 1:
        creator = {"creatorType": "author", "lastName": tokens[0]}
    else:
        creator = {
            "creatorType": "author",
            "firstName": " ".join(tokens[:-1]),
            "lastName": tokens[-1],
        }
    return normalize_author_name(creator, AMBIGUOUS_INITIAL_LAST)


def _is_address_like(token: str) -> bool:
    t = str(token or "").strip().lower()
    if not t:
        return True
    if t in COUNTRY_OR_REGION_TOKENS:
        return True
    if re.fullmatch(r"[0-9]{4,6}", t):
        return True
    if re.search(r"\b\d{4,6}\b", t):
        return True
    if re.fullmatch(r"[a-z]{1,3}\d{1,3}[a-z0-9\-]*", t):
        return True
    return False


def _strip_country_suffix(name: str) -> str:
    out = str(name or "").strip()
    low = out.lower()
    if "university" not in low:
        out = re.sub(r"\s*\(united states\)\s*$", "", out, flags=re.IGNORECASE).strip()
    return out


def _looks_like_institution(name: str) -> bool:
    low = str(name or "").lower()
    return any(k in low for k in INSTITUTION_KEYWORDS)


def _canonical_affiliation_name(raw_affiliation: str) -> str | None:
    """Reduce verbose affiliation strings to institution-level canonical labels."""
    raw = " ".join(str(raw_affiliation or "").replace(";", ",").split())
    if not raw:
        return None

    key = _affiliation_key(raw)
    if key in AFFILIATION_ALIAS_DYNAMIC:
        canonical = AFFILIATION_ALIAS_DYNAMIC[key]
        _register_full_affiliation(raw, canonical)
        return canonical
    if key in AFFILIATION_ALIAS_STATIC:
        canonical = _strip_country_suffix(AFFILIATION_ALIAS_STATIC[key])
        _register_alias(raw, canonical)
        _register_full_affiliation(raw, canonical)
        return canonical

    low = raw.lower()

    if "university of cincinnati" in low:
        _register_alias(raw, "University of Cincinnati")
        _register_full_affiliation(raw, "University of Cincinnati")
        return "University of Cincinnati"
    if "department of mechanical and materials engineering" in low and "cincinnati" in low:
        _register_alias(raw, "University of Cincinnati")
        _register_full_affiliation(raw, "University of Cincinnati")
        return "University of Cincinnati"
    if "imperial college london" in low:
        _register_alias(raw, "Imperial College London")
        _register_full_affiliation(raw, "Imperial College London")
        return "Imperial College London"
    if "centre national de la recherche scientifique" in low or re.search(r"\bcnrs\b", low):
        _register_alias(raw, "CNRS")
        _register_full_affiliation(raw, "CNRS")
        return "CNRS"
    if "paris sciences et lettres" in low or re.search(r"\bpsl\b", low):
        _register_alias(raw, "PSL University")
        _register_full_affiliation(raw, "PSL University")
        return "PSL University"
    if "ruhr" in low and "bochum" in low:
        _register_alias(raw, "Ruhr University Bochum")
        _register_full_affiliation(raw, "Ruhr University Bochum")
        return "Ruhr University Bochum"
    if "air force research laboratory" in low or re.search(r"\bafrl\b", low):
        _register_alias(raw, "Air Force Research Laboratory")
        _register_full_affiliation(raw, "Air Force Research Laboratory")
        return "Air Force Research Laboratory"

    parts = [p.strip(" .") for p in raw.split(",") if p.strip()]
    parts = [p for p in parts if not _is_address_like(p)]
    if not parts:
        return None

    institution_parts = [p for p in parts if _looks_like_institution(p)]
    candidate = institution_parts[0] if institution_parts else parts[0]

    cl = candidate.lower()
    if "university of cincinnati" in cl or ("cincinnati" in cl and "university" in cl):
        candidate = "University of Cincinnati"

    candidate = _strip_country_suffix(candidate)
    if not _looks_like_institution(candidate):
        return None

    _register_alias(raw, candidate)
    _register_full_affiliation(raw, candidate)
    return candidate


def _fetch_crossref_author_affiliations(
    doi: str,
    session: requests.Session,
    cache: dict,
 ) -> dict[str, set[str]]:
    norm = _normalize_doi(doi)
    if not norm:
        return {}
    if norm in cache:
        return cache[norm]

    out: dict[str, set[str]] = defaultdict(set)
    try:
        url = "https://api.crossref.org/works/" + urllib.parse.quote(norm, safe="")
        resp = session.get(
            url,
            timeout=15,
            headers={"User-Agent": "CARDLab/1.0 (mailto:paytonej@ucmail.uc.edu)"},
        )
        if resp.status_code != 200:
            cache[norm] = {}
            return {}

        message = resp.json().get("message", {})
        for author in message.get("author", []):
            given = str(author.get("given", "") or "").strip()
            family = str(author.get("family", "") or "").strip()
            canonical = _canonical_from_given_family(given, family)
            if not canonical:
                continue
            for aff in author.get("affiliation", []):
                name = _canonical_affiliation_name(aff.get("name", ""))
                if name:
                    out[canonical].add(name)

        cache[norm] = out
        return out
    except Exception:
        cache[norm] = {}
        return {}


def _fetch_crossref_work_affiliations(
    doi: str,
    session: requests.Session,
    cache: dict,
 ) -> set[str]:
    norm = _normalize_doi(doi)
    if not norm:
        return set()

    key = f"work::{norm}"
    if key in cache:
        return cache[key]

    out: set[str] = set()
    try:
        url = "https://api.crossref.org/works/" + urllib.parse.quote(norm, safe="")
        resp = session.get(
            url,
            timeout=15,
            headers={"User-Agent": "CARDLab/1.0 (mailto:paytonej@ucmail.uc.edu)"},
        )
        if resp.status_code != 200:
            cache[key] = set()
            return set()

        message = resp.json().get("message", {})
        for author in message.get("author", []):
            for aff in author.get("affiliation", []):
                name = _canonical_affiliation_name(aff.get("name", ""))
                if name:
                    out.add(name)
    except Exception:
        out = set()

    cache[key] = out
    return out


def _fetch_openalex_author_affiliations(
    doi: str,
    session: requests.Session,
    cache: dict,
 ) -> dict[str, set[str]]:
    norm = _normalize_doi(doi)
    if not norm:
        return {}
    if norm in cache:
        return cache[norm]

    out: dict[str, set[str]] = defaultdict(set)
    try:
        work_id = f"https://doi.org/{norm}"
        url = "https://api.openalex.org/works/" + urllib.parse.quote(work_id, safe="")
        resp = session.get(
            url,
            timeout=15,
            headers={"User-Agent": "CARDLab/1.0 (mailto:paytonej@ucmail.uc.edu)"},
            params={"mailto": "paytonej@ucmail.uc.edu"},
        )
        if resp.status_code != 200:
            cache[norm] = {}
            return {}

        message = resp.json()
        for authorship in message.get("authorships", []):
            display_name = ((authorship.get("author") or {}).get("display_name") or "").strip()
            canonical = _canonical_from_display(display_name)
            if not canonical:
                continue

            for inst in authorship.get("institutions", []) or []:
                inst_name = _canonical_affiliation_name(inst.get("display_name", ""))
                if inst_name:
                    out[canonical].add(inst_name)

        cache[norm] = out
        return out
    except Exception:
        cache[norm] = {}
        return {}


def _fetch_openalex_work_affiliations(
    doi: str,
    session: requests.Session,
    cache: dict,
 ) -> set[str]:
    norm = _normalize_doi(doi)
    if not norm:
        return set()

    key = f"work::{norm}"
    if key in cache:
        return cache[key]

    out: set[str] = set()
    try:
        work_id = f"https://doi.org/{norm}"
        url = "https://api.openalex.org/works/" + urllib.parse.quote(work_id, safe="")
        resp = session.get(
            url,
            timeout=15,
            headers={"User-Agent": "CARDLab/1.0 (mailto:paytonej@ucmail.uc.edu)"},
            params={"mailto": "paytonej@ucmail.uc.edu"},
        )
        if resp.status_code != 200:
            cache[key] = set()
            return set()

        message = resp.json()
        for authorship in message.get("authorships", []) or []:
            for inst in authorship.get("institutions", []) or []:
                inst_name = _canonical_affiliation_name(inst.get("display_name", ""))
                if inst_name:
                    out.add(inst_name)
    except Exception:
        out = set()

    cache[key] = out
    return out


def _fetch_datacite_author_affiliations(
    doi: str,
    session: requests.Session,
    cache: dict,
 ) -> dict[str, set[str]]:
    norm = _normalize_doi(doi)
    if not norm:
        return {}

    key = f"dc_author::{norm}"
    if key in cache:
        return cache[key]

    out: dict[str, set[str]] = defaultdict(set)
    try:
        url = "https://api.datacite.org/dois/" + urllib.parse.quote(norm, safe="")
        resp = session.get(url, timeout=15)
        if resp.status_code != 200:
            cache[key] = {}
            return {}

        attrs = ((resp.json().get("data") or {}).get("attributes") or {})
        creators = attrs.get("creators") or []
        for c in creators:
            given = str(c.get("givenName", "") or "").strip()
            family = str(c.get("familyName", "") or "").strip()
            canonical = _canonical_from_given_family(given, family)
            if not canonical:
                continue

            for aff in c.get("affiliation") or []:
                if isinstance(aff, dict):
                    raw_name = aff.get("name", "")
                else:
                    raw_name = str(aff)
                inst_name = _canonical_affiliation_name(raw_name)
                if inst_name:
                    out[canonical].add(inst_name)
    except Exception:
        out = defaultdict(set)

    cache[key] = out
    return out


def _fetch_datacite_work_affiliations(
    doi: str,
    session: requests.Session,
    cache: dict,
 ) -> set[str]:
    norm = _normalize_doi(doi)
    if not norm:
        return set()

    key = f"dc_work::{norm}"
    if key in cache:
        return cache[key]

    out: set[str] = set()
    try:
        url = "https://api.datacite.org/dois/" + urllib.parse.quote(norm, safe="")
        resp = session.get(url, timeout=15)
        if resp.status_code != 200:
            cache[key] = set()
            return set()

        attrs = ((resp.json().get("data") or {}).get("attributes") or {})
        creators = attrs.get("creators") or []
        for c in creators:
            for aff in c.get("affiliation") or []:
                if isinstance(aff, dict):
                    raw_name = aff.get("name", "")
                else:
                    raw_name = str(aff)
                inst_name = _canonical_affiliation_name(raw_name)
                if inst_name:
                    out.add(inst_name)
    except Exception:
        out = set()

    cache[key] = out
    return out


def _extract_creator_affiliations_from_zotero(data: dict) -> dict[str, set[str]]:
    out: dict[str, set[str]] = defaultdict(set)
    for creator in data.get("creators", []):
        canonical = normalize_author_name(creator, AMBIGUOUS_INITIAL_LAST)
        if not canonical:
            continue
        for key in ["affiliation", "institution", "university", "company"]:
            value = _canonical_affiliation_name(creator.get(key, ""))
            if value:
                out[canonical].add(value)
    return out


def _extract_affiliations_from_extra(extra: str) -> set[str]:
    extra = str(extra or "")
    if not extra.strip():
        return set()

    affiliations = set()
    pattern = re.compile(r"(?:^|\n)\s*Affiliations?\s*:\s*(.+)", re.IGNORECASE)
    for match in pattern.findall(extra):
        parts = re.split(r"[;|]", match)
        for part in parts:
            canonical = _canonical_affiliation_name(part)
            if canonical:
                affiliations.add(canonical)
    return affiliations


session = requests.Session()
doi_crossref_cache: dict[str, dict[str, set[str]]] = {}
doi_openalex_cache: dict[str, dict[str, set[str]]] = {}
doi_datacite_cache: dict[str, dict[str, set[str]]] = {}
paper_affiliations: list[set[str]] = []
doi_to_affiliations: dict[str, set[str]] = {}

source_stats = {
    "crossref_author_affiliation_hits": 0,
    "openalex_author_affiliation_hits": 0,
    "datacite_author_affiliation_hits": 0,
    "crossref_work_affiliation_hits": 0,
    "openalex_work_affiliation_hits": 0,
    "datacite_work_affiliation_hits": 0,
    "zotero_creator_affiliation_hits": 0,
    "zotero_extra_affiliation_hits": 0,
    "author_override_hits": 0,
    "papers_with_no_affiliations": 0,
    "papers_with_2plus_affiliations": 0,
}

for _, row in papers_df.iterrows():
    doi = row.get("doi", "")
    norm_doi = _normalize_doi(doi)
    data = row.get("raw_data", {}) or {}
    paper_authors = set(row.get("authors", []))

    aff_by_author: dict[str, set[str]] = defaultdict(set)

    crossref_map = _fetch_crossref_author_affiliations(doi, session, doi_crossref_cache)
    for author, affs in crossref_map.items():
        if author in paper_authors and affs:
            aff_by_author[author].update(affs)
    if any(crossref_map.get(a) for a in paper_authors):
        source_stats["crossref_author_affiliation_hits"] += 1

    openalex_map = _fetch_openalex_author_affiliations(doi, session, doi_openalex_cache)
    for author, affs in openalex_map.items():
        if author in paper_authors and affs:
            aff_by_author[author].update(affs)
    if any(openalex_map.get(a) for a in paper_authors):
        source_stats["openalex_author_affiliation_hits"] += 1

    datacite_map = _fetch_datacite_author_affiliations(doi, session, doi_datacite_cache)
    for author, affs in datacite_map.items():
        if author in paper_authors and affs:
            aff_by_author[author].update(affs)
    if any(datacite_map.get(a) for a in paper_authors):
        source_stats["datacite_author_affiliation_hits"] += 1

    crossref_work_affs = _fetch_crossref_work_affiliations(doi, session, doi_crossref_cache)
    if crossref_work_affs:
        source_stats["crossref_work_affiliation_hits"] += 1

    openalex_work_affs = _fetch_openalex_work_affiliations(doi, session, doi_openalex_cache)
    if openalex_work_affs:
        source_stats["openalex_work_affiliation_hits"] += 1

    datacite_work_affs = _fetch_datacite_work_affiliations(doi, session, doi_datacite_cache)
    if datacite_work_affs:
        source_stats["datacite_work_affiliation_hits"] += 1

    creator_map = _extract_creator_affiliations_from_zotero(data)
    for author, affs in creator_map.items():
        if author in paper_authors and affs:
            aff_by_author[author].update(affs)
    if any(creator_map.get(a) for a in paper_authors):
        source_stats["zotero_creator_affiliation_hits"] += 1

    extra_affs = _extract_affiliations_from_extra(data.get("extra", ""))
    if extra_affs:
        for author in paper_authors:
            aff_by_author[author].update(extra_affs)
        source_stats["zotero_extra_affiliation_hits"] += 1

    used_override = False
    for author in paper_authors:
        override_affs = {_canonical_affiliation_name(x) for x in AUTHOR_AFFILIATION_OVERRIDES.get(author, set())}
        override_affs = {x for x in override_affs if x}
        if override_affs:
            aff_by_author[author].update(override_affs)
            used_override = True
    if used_override:
        source_stats["author_override_hits"] += 1

    affiliations = {
        aff
        for author in paper_authors
        for aff in aff_by_author.get(author, set())
        if aff and len(str(aff).strip()) > 2
    }

    affiliations.update(crossref_work_affs)
    affiliations.update(openalex_work_affs)
    affiliations.update(datacite_work_affs)
    affiliations = {a for a in affiliations if a and len(str(a).strip()) > 2}

    if not affiliations:
        source_stats["papers_with_no_affiliations"] += 1
    if len(affiliations) >= 2:
        source_stats["papers_with_2plus_affiliations"] += 1

    paper_affiliations.append(affiliations)
    if norm_doi:
        doi_to_affiliations[norm_doi] = set(affiliations)

AG = nx.Graph()
for affiliations in paper_affiliations:
    aff_list = sorted(affiliations)
    for aff in aff_list:
        if aff not in AG:
            AG.add_node(aff, papers=0)
        AG.nodes[aff]["papers"] += 1

    for a, b in combinations(aff_list, 2):
        if AG.has_edge(a, b):
            AG[a][b]["weight"] += 1
        else:
            AG.add_edge(a, b, weight=1)

card_seed_affiliations = set()
for idx, row in papers_df.iterrows():
    if any(a in CARD_CORE_AUTHORS_CANONICAL for a in row["authors"]):
        card_seed_affiliations.update(paper_affiliations[idx])

if not card_seed_affiliations:
    card_seed_affiliations = set(CARD_SEED_AFFILIATIONS)

card_seed_affiliations.add("University of Cincinnati")

present_seeds = {s for s in card_seed_affiliations if s in AG.nodes}
if not present_seeds:
    present_seeds = {s for s in CARD_SEED_AFFILIATIONS if s in AG.nodes}

if not present_seeds and AG.number_of_nodes() > 0:
    present_seeds = {
        n for n, _ in sorted(AG.degree(weight="weight"), key=lambda x: x[1], reverse=True)[:3]
    }

keep_nodes = set()
for seed in present_seeds:
    lengths = nx.single_source_shortest_path_length(AG, seed, cutoff=1)
    keep_nodes.update(lengths.keys())

AH = AG.subgraph(keep_nodes).copy()
aff_weighted_degree = dict(AH.degree(weight="weight"))

print(f"Affiliation graph (full): {AG.number_of_nodes()} nodes, {AG.number_of_edges()} edges")
print(f"Affiliation graph (1-hop CARD-centered): {AH.number_of_nodes()} nodes, {AH.number_of_edges()} edges")
print(f"Seed affiliations used: {sorted(present_seeds)}")
print("Affiliation source diagnostics:", source_stats)
print(f"Dynamic aliases learned this run: {len(AFFILIATION_ALIAS_DYNAMIC)}")
if AG.number_of_nodes() > 0:
    top_aff = sorted(AG.degree(weight="weight"), key=lambda x: x[1], reverse=True)[:20]
    print("Top affiliations by weighted degree:", [x[0] for x in top_aff])

if TARGET_DIAGNOSTIC_DOI in doi_to_affiliations:
    print(f"Affiliations for DOI {TARGET_DIAGNOSTIC_DOI}: {sorted(doi_to_affiliations[TARGET_DIAGNOSTIC_DOI])}")
else:
    print(f"DOI {TARGET_DIAGNOSTIC_DOI} not present in resolved multi-author paper set.")


Affiliation graph (full): 15 nodes, 33 edges
Affiliation graph (1-hop CARD-centered): 15 nodes, 33 edges
Seed affiliations used: ['Air Force Research Laboratory', 'CNRS', 'Case Western Reserve University', 'Imperial College London', 'Institut de Recherche de Chimie Paris', 'Lehigh University', 'Materials Resources', 'PSL University', 'Ruhr University Bochum', 'The Ohio State University', 'University of Cincinnati', 'University of Dayton', 'University of Florida', 'Wright State University', 'Wright-Patterson Air Force Base']
Affiliation source diagnostics: {'crossref_author_affiliation_hits': 1, 'openalex_author_affiliation_hits': 19, 'datacite_author_affiliation_hits': 0, 'crossref_work_affiliation_hits': 1, 'openalex_work_affiliation_hits': 19, 'datacite_work_affiliation_hits': 0, 'zotero_creator_affiliation_hits': 0, 'zotero_extra_affiliation_hits': 0, 'author_override_hits': 23, 'papers_with_no_affiliations': 0, 'papers_with_2plus_affiliations': 11}
Dynamic aliases learned this run:

In [7]:
#| echo: false
#| eval: true
#| output: true

if AH.number_of_nodes() == 0:
    print("No affiliation network could be formed from available author-affiliation metadata.")
else:
    pos_a = nx.spring_layout(AH, dim=3, k=1.1 / math.sqrt(max(1, AH.number_of_nodes())), seed=42)

    edge_x_a, edge_y_a, edge_z_a = [], [], []
    for u, v, _ in AH.edges(data=True):
        x0, y0, z0 = pos_a[u]
        x1, y1, z1 = pos_a[v]
        edge_x_a += [x0, x1, None]
        edge_y_a += [y0, y1, None]
        edge_z_a += [z0, z1, None]

    edge_trace_a = go.Scatter3d(
        x=edge_x_a,
        y=edge_y_a,
        z=edge_z_a,
        mode="lines",
        line=dict(width=4, color="rgba(120,120,120,0.6)"),
        hoverinfo="none",
    )

    node_x_a, node_y_a, node_z_a = [], [], []
    node_text_a, node_size_a, node_color_a = [], [], []
    for n, attrs in AH.nodes(data=True):
        x, y, z = pos_a[n]
        wdeg = aff_weighted_degree.get(n, 0)
        full_address = AFFILIATION_FULL_ADDRESS.get(n, n)

        node_x_a.append(x)
        node_y_a.append(y)
        node_z_a.append(z)
        node_size_a.append(7 + 3 * math.sqrt(max(1, wdeg)))
        node_color_a.append("#d62728" if n in present_seeds else "#2ca02c")
        node_text_a.append(
            f"<b>{n}</b><br>"
            f"Full address: {full_address}<br>"
            f"Papers represented: {attrs.get('papers', 0)}<br>"
            f"Weighted degree: {wdeg}"
        )

    node_trace_a = go.Scatter3d(
        x=node_x_a,
        y=node_y_a,
        z=node_z_a,
        mode="markers+text",
        text=list(AH.nodes()),
        textposition="top center",
        hovertemplate="%{customdata}<extra></extra>",
        customdata=node_text_a,
        marker=dict(size=node_size_a, color=node_color_a, line=dict(width=0.8, color="white"), opacity=0.95),
    )


In [8]:

    fig_a = go.Figure(data=[edge_trace_a, node_trace_a])
    fig_a.update_layout(
        title="Affiliation Collaboration Network",
        showlegend=False,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=0, r=0, t=50, b=0),
        scene=dict(
            bgcolor="white",
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
            zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title="", showbackground=False),
            dragmode="orbit",
        ),
    )
    # Add very slow auto-rotation by animating the camera eye around the z-axis.
    n_frames_a = 240
    radius_a = 1.9
    z_eye_a = 0.9
    fig_a.frames = [
        go.Frame(layout=dict(scene_camera=dict(eye=dict(
            x=radius_a * math.cos(2 * math.pi * i / n_frames_a),
            y=radius_a * math.sin(2 * math.pi * i / n_frames_a),
            z=z_eye_a,
        ))))
        for i in range(n_frames_a)
    ]
    fig_a.update_layout(scene_camera=dict(eye=dict(x=radius_a, y=0, z=z_eye_a)))
    fig_a.show()

In [9]:
#| echo: false
#| eval: true
#| output: true


# Top non-CARD co-authors and affiliations (great_tables output).
from collections import Counter, defaultdict

try:
    from great_tables import GT
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "great-tables", "-q"])
    from great_tables import GT

non_core_papers_with_card = Counter()
non_core_affiliations = defaultdict(set)

for _, row in papers_df.iterrows():
    authors = set(row.get("authors", []))
    core_authors_on_paper = authors & CARD_CORE_AUTHORS_CANONICAL
    non_core_authors_on_paper = authors - CARD_CORE_AUTHORS_CANONICAL

    if not core_authors_on_paper or not non_core_authors_on_paper:
        continue

    doi = row.get("doi", "")
    norm_doi = _normalize_doi(doi) if "_normalize_doi" in globals() else str(doi or "").strip().lower()
    data = row.get("raw_data", {}) or {}

    aff_by_author = defaultdict(set)

    if "_fetch_crossref_author_affiliations" in globals():
        try:
            crossref_map = _fetch_crossref_author_affiliations(doi, session, doi_crossref_cache)
            for a, affs in crossref_map.items():
                if a in non_core_authors_on_paper and affs:
                    aff_by_author[a].update(affs)
        except Exception:
            pass

    if "_fetch_openalex_author_affiliations" in globals():
        try:
            openalex_map = _fetch_openalex_author_affiliations(doi, session, doi_openalex_cache)
            for a, affs in openalex_map.items():
                if a in non_core_authors_on_paper and affs:
                    aff_by_author[a].update(affs)
        except Exception:
            pass

    if "_fetch_datacite_author_affiliations" in globals():
        try:
            datacite_map = _fetch_datacite_author_affiliations(doi, session, doi_datacite_cache)
            for a, affs in datacite_map.items():
                if a in non_core_authors_on_paper and affs:
                    aff_by_author[a].update(affs)
        except Exception:
            pass

    if "_extract_creator_affiliations_from_zotero" in globals():
        try:
            creator_map = _extract_creator_affiliations_from_zotero(data)
            for a, affs in creator_map.items():
                if a in non_core_authors_on_paper and affs:
                    aff_by_author[a].update(affs)
        except Exception:
            pass

    for author in sorted(non_core_authors_on_paper):
        non_core_papers_with_card[author] += 1
        if author in aff_by_author and aff_by_author[author]:
            non_core_affiliations[author].update(aff_by_author[author])
        elif "doi_to_affiliations" in globals() and norm_doi in doi_to_affiliations:
            non_core_affiliations[author].update(doi_to_affiliations[norm_doi])

# Explicit correction requested by user: A. Pilchak is with Pratt & Whitney.
for pilchak_name in ["A. Pilchak", "Aaron Pilchak", "Aaron J. Pilchak"]:
    if pilchak_name in non_core_papers_with_card:
        non_core_affiliations[pilchak_name] = {"Pratt & Whitney"}

rows = []
for author, with_card_count in non_core_papers_with_card.items():
    affiliations = sorted(non_core_affiliations.get(author, set()))
    rows.append(
        {
            "coauthor": author,
            "# Papers with CARD Group": with_card_count,
            "affiliations": "; ".join(affiliations) if affiliations else "Unknown",
        }
    )

top_non_core_summary = (
    pd.DataFrame(rows)
    .sort_values(["# Papers with CARD Group", "coauthor"], ascending=[False, True])
    .head(10)
    .reset_index(drop=True)
    if rows
    else pd.DataFrame(columns=["coauthor", "# papers", "affiliation"])
)


(GT(top_non_core_summary).tab_header(
    title="Top 10 Co-Authors",
    subtitle="Ranked by number of papers with CARD group",
)
.cols_align(align="center", columns="# Papers with CARD Group"))

GT(_tbl_data=      coauthor  # Papers with CARD Group  \
0   M. Steiner                         4   
1   N. Simpson                         3   
2    V. Miller                         3   
3   G. Eggeler                         2   
4  M. Gonzales                         2   
5    O. Senkov                         2   
6       Y. Lee                         2   
7      A. Amin                         1   
8     A. Gerlt                         1   
9   A. Pilchak                         1   

                                        affiliations  
0                           University of Cincinnati  
1                           University of Cincinnati  
2                              University of Florida  
3                             Ruhr University Bochum  
4  Air Force Research Laboratory; Wright-Patterso...  
5  Air Force Research Laboratory; Materials Resou...  
6                              University of Florida  
7                               University of Dayton  
8                          The Ohio State University  
9                                    Pratt & Whitney  , _body=<great_tables._gt_data.Body object at 0x126550350>, _boxhead=Boxhead([ColInfo(var='coauthor', type=<ColInfoTypeEnum.default: 1>, column_label='coauthor', column_align='left', column_width=None), ColInfo(var='# Papers with CARD Group', type=<ColInfoTypeEnum.default: 1>, column_label='# Papers with CARD Group', column_align='center', column_width=None), ColInfo(var='affiliations', type=<ColInfoTypeEnum.default: 1>, column_label='affiliations', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1252ccc10>, _spanners=Spanners([]), _heading=Heading(title='Top 10 Co-Authors', subtitle='Ranked by number of papers with CARD group', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x126550890>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x126550690>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x126550810>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_wi